### 验证时间管理能力 计算能力 能不能在长期阶段影响ELO

In [1]:
import pandas as pd
import json
import ast
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import seaborn as sns
import numpy as np
from scipy.optimize import curve_fit
from tqdm import tqdm
from tqdm.auto import tqdm
tqdm.pandas()
import os
import math
import psutil
import gc
import chess
import chess.engine
import chess.svg
from IPython.display import display, HTML
import subprocess
import time
from pathlib import Path
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
import matplotlib.patches as patches
import sqlite3
from matplotlib.patches import ConnectionPatch, Rectangle
from functools import reduce

db_path = r"C:\sqlite3\chess.db"
table = 'games'

从数据库提取 ELO 并计算周涨跌 (目标变量)

In [2]:
conn = sqlite3.connect(db_path)
df_games = pd.read_sql('SELECT uid, Date, White, Black, WhiteElo, BlackElo FROM games', conn)
conn.close()

# 2. 拆分黑白方并合并为选手时间序列
white_players = df_games[['uid', 'White', 'Date', 'WhiteElo']].rename(columns={'White': 'Player', 'Date': 'Date', 'WhiteElo': 'ELO'})
black_players = df_games[['uid', 'Black', 'Date', 'BlackElo']].rename(columns={'Black': 'Player', 'Date': 'Date', 'BlackElo': 'ELO'})
player_history = pd.concat([white_players, black_players])
player_history['Player'] = player_history['Player'].str.lower().str.strip()
player_history['Date'] = pd.to_datetime(player_history['Date'])

# 及时清理临时变量
del df_games, white_players, black_players
gc.collect()

# 3. 筛选活跃选手 (比赛 > 200场)
player_counts = player_history.groupby('Player').size()
active_players = player_counts[player_counts >= 200].index
active_history = player_history[player_history['Player'].isin(active_players)].copy()

# 4. 计算全局连续周 (Week_Idx) 并构建周度目标变量
min_date = active_history['Date'].min()
active_history['Week_Idx'] = (active_history['Date'] - min_date).dt.days // 7

weekly_target = active_history.groupby(['Player', 'Week_Idx']).agg(
    Weekly_Avg_ELO=('ELO', 'mean'),
    Games_This_Week=('ELO', 'count')
).reset_index().sort_values(['Player', 'Week_Idx'])

# 计算下周的连续分差 (ELO_Change)
weekly_target['Next_Week_Idx'] = weekly_target.groupby('Player')['Week_Idx'].shift(-1)
weekly_target['Next_Weekly_ELO'] = weekly_target.groupby('Player')['Weekly_Avg_ELO'].shift(-1)

weekly_target['ELO_Change'] = np.where(
    (weekly_target['Next_Week_Idx'] - weekly_target['Week_Idx']) == 1,
    weekly_target['Next_Weekly_ELO'] - weekly_target['Weekly_Avg_ELO'],
    np.nan
)

# 清理内存，保留黄金目标表 weekly_target
del player_history
gc.collect()

17

In [3]:
weekly_target.head()

,Player,Week_Idx,Weekly_Avg_ELO,Games_This_Week,Next_Week_Idx,Next_Weekly_ELO,ELO_Change
0,4empechement,63,2477.000000,11,70.0,2401.363636,NaN
1,4empechement,70,2401.363636,11,71.0,2373.727273,-27.636364
2,4empechement,71,2373.727273,11,77.0,2386.000000,NaN
3,4empechement,77,2386.000000,11,80.0,2406.818182,NaN
4,4empechement,80,2406.818182,11,86.0,2458.272727,NaN


加载df_moves 大表

In [4]:
# 1. 读取棋步大表
df_moves = pd.read_parquet(r'..\df_moves6.parquet')

# 2. 仅保留活跃选手，转小写
active_set = set(active_players)
df_moves['Player'] = df_moves['Player'].str.lower()
df_moves_active = df_moves[df_moves['Player'].isin(active_set)].copy()

del df_moves
gc.collect()

# 3. 内存瘦身术 (将占用减半，极度关键)
cols_to_fix = ['Is_Optimal', 'Δi', 'Is_Blunder', 'Is_Error', 'Cog_Speed', 'Remain_Time']
for col in cols_to_fix:
    df_moves_active[col] = pd.to_numeric(df_moves_active[col], errors='coerce').astype('float32') # 强制转为 float32

# 修复异常值
df_moves_active[cols_to_fix] = df_moves_active[cols_to_fix].replace([np.inf, -np.inf], np.nan)

计算时间管理分数

In [5]:
# 1. 提取微型数据框并预过滤
mask = df_moves_active['Progress'] >= 0.5
slim_df = df_moves_active.loc[mask, ['uid', 'Player', 'Progress', 'Move_Idx', 'Remain_Time']].copy()

# 2. 计算残差
time_baseline = df_moves_active.groupby('Move_Idx')['Remain_Time'].mean().astype('float32')
slim_df['Expected_Remain'] = slim_df['Move_Idx'].map(time_baseline)
slim_df['Time_Residual'] = slim_df['Remain_Time'] - slim_df['Expected_Remain']

progress_stones = [0.6, 0.7, 0.8, 0.9]
tm_scores_list = []

# 3. 迭代计算每个节点的得分，存入列表
for p in progress_stones:
    # 【修正处】先将绝对值距离算成一列 'dist'
    slim_df['dist'] = (slim_df['Progress'] - p).abs()
    
    # 然后再按照 'dist' 列去找最小值对应的索引
    idx = slim_df.groupby(['uid', 'Player'])['dist'].idxmin()
    
    # 提取这些行
    target_rows = slim_df.loc[idx].copy()
    
    # 计算得分，统一列名 node_score，方便后续直接纵向求和
    target_rows['node_score'] = (target_rows['Time_Residual'] / 180.0) * target_rows['Progress']
    tm_scores_list.append(target_rows[['uid', 'Player', 'node_score']])

# 4. 纵向合并代替横向合并 (避免由于 merge 造成的内存爆炸)
all_nodes_df = pd.concat(tm_scores_list, ignore_index=True)

# 聚合出最终的局级时间管理分
game_level_tm = all_nodes_df.groupby(['uid', 'Player'])['node_score'].sum().reset_index()
game_level_tm.rename(columns={'node_score': 'Time_Management_Score'}, inplace=True)
game_level_tm['Time_Management_Score'] = game_level_tm['Time_Management_Score'].round(2)

# 及时清理内存
del slim_df, time_baseline, tm_scores_list, all_nodes_df
gc.collect()

# 5. 将计算好的局级分数左连接合并回 df_moves_active
df_moves_active = df_moves_active.merge(game_level_tm, on=['uid', 'Player'], how='left')
del game_level_tm
gc.collect()

print("Step 3: 时间管理分计算完成！")

Step 3: 时间管理分计算完成！


In [6]:
# 1. 同步周索引 (Week_Idx)
uid_week_map = active_history[['uid', 'Date']].drop_duplicates('uid')
uid_week_map['Week_Idx'] = (uid_week_map['Date'] - min_date).dt.days // 7
df_moves_active = df_moves_active.merge(uid_week_map[['uid', 'Week_Idx']], on='uid', how='left')

# 2. 步级特征聚合
player_week_features = df_moves_active.groupby(['Player', 'Week_Idx']).agg({
    'Is_Optimal': 'mean',
    'Δi': 'mean',
    'Is_Blunder': 'mean',
    'Is_Error': 'mean',
    'Cog_Speed': 'mean',
    'uid': 'count' 
}).rename(columns={'uid': 'move_count'}).reset_index()

# 3. 局级特征 (Time_Management_Score) 聚合
# 先去重，保证每局游戏的时间管理分只算一次
game_unique_scores = df_moves_active[['uid', 'Player', 'Week_Idx', 'Time_Management_Score']].drop_duplicates(subset=['uid', 'Player'])
weekly_tm_avg = game_unique_scores.groupby(['Player', 'Week_Idx'])['Time_Management_Score'].mean().reset_index()

# 合并步级与局级特征
player_week_features = player_week_features.merge(weekly_tm_avg, on=['Player', 'Week_Idx'], how='left')

# 4. 剔除全空的噪音行
feature_cols = ['Is_Optimal', 'Δi', 'Is_Blunder', 'Is_Error', 'Cog_Speed', 'Time_Management_Score']
player_week_features = player_week_features.dropna(subset=feature_cols, how='all')

# 5. 【大一统】合并目标变量 (标签 y)
final_dataset = player_week_features.merge(
    weekly_target[['Player', 'Week_Idx', 'Weekly_Avg_ELO', 'ELO_Change']], 
    on=['Player', 'Week_Idx'], 
    how='inner'
)

print(final_dataset.head())

         Player  Week_Idx  Is_Optimal          Δi  Is_Blunder  Is_Error  \
0  4empechement        63    0.500000   77.541832    0.043825  0.087649   
1  4empechement        70    0.520792  128.621780    0.037624  0.081188   
2  4empechement        71    0.645652   74.480438    0.028261  0.043478   
3  4empechement        77    0.536585   77.416855    0.031042  0.066519   
4  4empechement        80    0.567500  113.712502    0.042500  0.070000   

   Cog_Speed  move_count  Time_Management_Score  Weekly_Avg_ELO  ELO_Change  
0   0.433976         502              -0.531818     2477.000000         NaN  
1   0.458203         505              -0.020909     2401.363636  -27.636364  
2   0.461216         460              -0.116364     2373.727273         NaN  
3   0.434357         451              -0.389091     2386.000000         NaN  
4   0.445507         400              -0.508182     2406.818182         NaN  


In [10]:
final_dataset.head()

,Player,Week_Idx,Is_Optimal,Δi,Is_Blunder,Is_Error,Cog_Speed,move_count,Time_Management_Score,Weekly_Avg_ELO,ELO_Change
0,4empechement,63,0.500000,77.541832,0.043825,0.087649,0.433976,502,-0.531818,2477.000000,NaN
1,4empechement,70,0.520792,128.621780,0.037624,0.081188,0.458203,505,-0.020909,2401.363636,-27.636364
2,4empechement,71,0.645652,74.480438,0.028261,0.043478,0.461216,460,-0.116364,2373.727273,NaN
3,4empechement,77,0.536585,77.416855,0.031042,0.066519,0.434357,451,-0.389091,2386.000000,NaN
4,4empechement,80,0.567500,113.712502,0.042500,0.070000,0.445507,400,-0.508182,2406.818182,NaN


In [11]:
%whos

Variable               Type                          Data/Info
--------------------------------------------------------------
ConnectionPatch        type                          <class 'matplotlib.patches.ConnectionPatch'>
HTML                   type                          <class 'IPython.core.display.HTML'>
Path                   type                          <class 'pathlib.Path'>
Rectangle              type                          <class 'matplotlib.patches.Rectangle'>
active_history         DataFrame                                  uid         <...>1659334 rows x 5 columns]
active_players         Index                         Index(['4empechement', 'a<...>me='Player', length=2659)
active_set             set                           {'alexandra kosteniuk', '<...>uria', 'mikhail chernov'}
ast                    module                        <module 'ast' from 'c:\\U<...>\Python312\\Lib\\ast.py'>
chess                  module                        <module 'chess' from 'c:\<...>

In [12]:
del mask, idx, cols_to_fix
del uid_week_map
del active_history
del target_rows, game_unique_scores
gc.collect()

28

In [18]:
final_dataset.to_parquet('skill-elo.parquet')